<a href="https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring, built on top of a classification model.** The reviewer's real need is a ranked list —
"which ~50 pages do I look at this week?" — so the deliverable is a **priority score** per page,
not a hard yes/no label. Under the hood, the score comes from a classifier's predicted
probability: I train a model to estimate P(page is declining), then sort pages by that
probability to get the queue. This matches the lane's action directly: a reviewer works down a
ranked list until their weekly capacity runs out, so what matters is whether the *top* of the
list is right (precision at the cutoff), not whether every page in the dataset is labeled
correctly.

It is not clustering (I'm not looking for undiscovered groups of pages) and it is not pure
ranking in the search-results sense (no pairwise "A before B" preference data) — it's a scoring
problem wearing a classifier's clothes.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` — 1 when `trend_direction == "down"`, else 0. `trend_direction`
itself is a defined rule: it compares `impressions_last_30d` vs `impressions_prev_30d` and calls
it `down` when impressions fell more than 20%.

**Honest framing — this is a proxy, not a fully observed future outcome.** Per the
`framing-ml-problems` skill, a target should ideally be an outcome *observed in a later time
window*, not something defined by a rule. `is_declining_label` is a **same-window** proxy: it's
computed from the same 90-day export I'd use for features, using a threshold someone chose
(±20%). That's good enough to build and validate the workflow now, but it means I'm currently
learning "does this page match the down-trend rule," not "will this page keep declining next
month." A stronger version of this lane (weeks 3+, full warehouse) would define the label from a
**future** window — e.g., did impressions fall in the 30 days *after* the feature window — using
only prior-period signals as features. I'm flagging that gap now rather than overclaiming later.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** The reviewer can only manually check about 50 pages a week (my capacity
assumption from Week 1), so the only thing that matters is: of the top 50 pages the model ranks
highest, how many are actually declining? That's Precision@K with K=50 — it directly mirrors the
action (a fixed-size weekly review queue) rather than a generic score like accuracy, which would
reward the model for correctly ignoring thousands of already-fine pages nobody was going to look
at anyway.

I'm not using ROC-AUC or plain accuracy as the *headline* number because neither one answers the
reviewer's actual question ("is my Monday-morning list worth trusting?") — I'll still look at them
as secondary diagnostics.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), aggregated over a trailing 90-day window, for one
of 32 pseudonymized clients (`client_id`). Loading the starter slice below and showing it as an
actual dataframe, plus what the target column looks like on this data.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Trisha108-hub/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows (one row = one content page): {df.shape[0]:,}")
print(f"Distinct clients: {df['client_id'].nunique()}")

# The unit of analysis, as a real dataframe: one row per page, key columns visible.
unit_view = df[[
    "content_id", "client_id", "content_type", "impressions_90d",
    "clicks_90d", "avg_position", "trend_pct", "trend_direction",
]]
unit_view.head(10)



Rows (one row = one content page): 30,000
Distinct clients: 32


,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,-60.9,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,-34.7,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,8.5,-38.9,down
6,content_9a34b442b552,client_8722616204,keyword article,20,0,7.0,-92.3,down
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,21.2,0.6,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,46.0,-58.8,down
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,4.9,-29.2,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter pipeline already answers this with numbers, not just an opinion: a transparent
hand-written rule ("flag pages where CTR/position/impressions look weak") reaches Precision@50 ≈
0.24, while a random forest trained on the same features reaches ≈ 0.68–0.74 — roughly a 3x lift
at picking the right pages. The gap exists because "worth reviewing" isn't governed by any single
threshold: a page can be declining because of a fresh-content cliff, a stale keyword losing search
volume, poor engagement despite decent traffic, or a shift toward AI-referred sessions — and the
same raw number (e.g. low CTR) means something different depending on `avg_position`,
`search_volume`, and `content_type` at the same time. A rule-writer has to either pick one signal
and miss the rest, or hand-stack dozens of nested if-statements and still miss interactions. A
model can weigh many correlated, non-linear signals at once and learn which *combinations* matter
— that's the "too messy for an if-statement" test the skill describes, and it's borne out by the
gap between the baseline rule and the model on this exact data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

# Show the concrete evidence: baseline rule vs model on the same Precision@50 metric.
if not os.path.exists("outputs/model_results.json"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
print(f"Random forest       Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
print(f"Model beats the fixed rule by {rf/base:.1f}x on the metric that matters for this lane")

# Sketch the target column itself: distribution + a peek at what triggers it.
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print()
print("Target column value counts:")
print(df["is_declining_label"].value_counts(normalize=True).rename("share"))
df[["content_id", "trend_pct", "trend_direction", "is_declining_label"]].sample(8, random_state=1)


Hand-written rule  Precision@50: 0.240  (~12 of top 50 correct)
Random forest       Precision@50: 0.740  (~37 of top 50 correct)
Model beats the fixed rule by 3.1x on the metric that matters for this lane

Target column value counts:
is_declining_label
1    0.542067
0    0.457933
Name: share, dtype: float64


,content_id,trend_pct,trend_direction,is_declining_label
10747,content_27441bff4885,-41.4,down,1
12573,content_6352ef618f56,-90.7,down,1
29676,content_5f43c51e146a,-100.0,down,1
8856,content_4e64474fabfa,-44.7,down,1
21098,content_d0b2b5a0ef58,89.2,up,0
17458,content_193bcf42ff80,-29.9,down,1
1476,content_6e792cf3ce56,-25.0,down,1
5120,content_cb3b002631a6,-5.0,stable,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.